# Farm Risk Demo

This notebook walks through a simple agricultural finance workflow using `ag-finance-toolkit`.
It is designed to feel demo-ready even without live API access: the core metrics and Monte Carlo sections run entirely from a realistic sample farm, and the connector examples are included as optional extensions.


## 1. Define A Sample Farm

We start with a stylized Iowa corn and soybean operation. The assumptions below are explicit so the resulting financial and risk outputs are easy to interpret.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from agfin import calculate_all, simulate_farm_risk
from agfin.risk import cost_distribution, price_distribution, yield_distribution
from agfin.schemas import CropMixItem, FarmInputs

farm = FarmInputs(
    total_acres=500.0,
    crop_mix=[
        CropMixItem(crop="Corn", acres=300.0, expected_yield=190.0, expected_price=4.8),
        CropMixItem(crop="Soybeans", acres=200.0, expected_yield=55.0, expected_price=12.0),
    ],
    operating_costs=250_000.0,
    fixed_costs=80_000.0,
    debt_obligations=60_000.0,
    working_capital=100_000.0,
    current_liabilities=40_000.0,
    total_assets=1_500_000.0,
    total_liabilities=600_000.0,
    net_worth=900_000.0,
    insurance_income=5_000.0,
    govt_payments=10_000.0,
    interest_expense=25_000.0,
    depreciation=45_000.0,
    owner_draws=30_000.0,
)

farm


In [ ]:
farm_profile = pd.DataFrame(
    [
        {"category": "Acres", "value": farm.total_acres, "notes": "Total operated acres"},
        {"category": "Operating costs", "value": farm.operating_costs, "notes": "Annual operating spend ($)"},
        {"category": "Fixed costs", "value": farm.fixed_costs, "notes": "Annual fixed cost burden ($)"},
        {"category": "Debt obligations", "value": farm.debt_obligations, "notes": "Annual debt service ($)"},
        {"category": "Working capital", "value": farm.working_capital, "notes": "Liquidity buffer ($)"},
    ]
)

farm_profile


## 2. Calculate Core Financial Metrics

The package exposes a schema-aware `calculate_all(...)` helper that translates validated inputs into liquidity, solvency, profitability, and repayment outputs.


In [ ]:
metrics = calculate_all(farm)

metrics_table = pd.DataFrame(
    [
        ["Working capital", metrics.working_capital, "$", "Short-term liquidity cushion"],
        ["Current ratio", metrics.current_ratio, "x", "Current assets divided by current liabilities"],
        ["Debt-to-asset", metrics.debt_to_asset, "ratio", "Leverage burden"],
        ["Equity ratio", metrics.equity_ratio, "ratio", "Share of assets financed by equity"],
        ["Net farm income", metrics.net_farm_income, "$", "Simplified annual earnings"],
        ["Operating margin", metrics.operating_margin, "ratio", "Income as a share of revenue"],
        ["Operating expense ratio", metrics.operating_expense_ratio, "ratio", "Operating costs as a share of revenue"],
        ["DSCR", metrics.dscr, "x", "Debt-service coverage ratio"],
    ],
    columns=["metric", "value", "unit", "interpretation"],
)

metrics_table["value"] = metrics_table.apply(
    lambda row: f"${row['value']:,.0f}" if row["unit"] == "$" else f"{row['value']:.2f}",
    axis=1,
)

metrics_table


## 3. Simulate Income And Repayment Risk

We now layer uncertainty on top of the base farm assumptions. The Monte Carlo model applies multiplicative shocks to yield, price, and operating costs, then summarizes the resulting income and DSCR distributions.


In [ ]:
yield_cv = 0.12
price_cv = 0.15
cost_cv = 0.08
runs = 10_000
seed = 42

risk = simulate_farm_risk(
    farm,
    yield_cv=yield_cv,
    price_cv=price_cv,
    cost_cv=cost_cv,
    runs=runs,
    seed=seed,
)

risk_table = pd.DataFrame(
    [
        ["Mean income", risk.mean_income, "$", "Average simulated net income"],
        ["P10 income", risk.p10_income, "$", "Income exceeded in 90% of simulations"],
        ["P50 income", risk.p50_income, "$", "Median simulated income"],
        ["P90 income", risk.p90_income, "$", "Upper-end income scenario"],
        ["Mean DSCR", risk.mean_dscr, "x", "Average debt-service coverage"],
        ["P10 DSCR", risk.p10_dscr, "x", "Downside debt-service coverage"],
        ["Default probability", risk.default_probability, "%", "Share of runs where DSCR falls below 1.0"],
        ["Worst-case shortfall", risk.worst_case_shortfall, "$", "Largest debt-service gap in the simulation"],
    ],
    columns=["metric", "value", "unit", "interpretation"],
)

risk_table["value"] = risk_table.apply(
    lambda row: (
        f"${row['value']:,.0f}" if row["unit"] == "$" else
        f"{100 * row['value']:.1f}%" if row["unit"] == "%" else
        f"{row['value']:.2f}"
    ),
    axis=1,
)

risk_table


In [ ]:
rng = np.random.default_rng(seed)

yield_multiplier = yield_distribution(1.0, yield_cv).rvs(size=runs, random_state=rng)
price_multiplier = price_distribution(1.0, price_cv).rvs(size=runs, random_state=rng)
cost_multiplier = np.maximum(
    cost_distribution(1.0, cost_cv).rvs(size=runs, random_state=rng),
    0.0,
)

base_revenue = sum(item.acres * item.expected_yield * item.expected_price for item in farm.crop_mix)
income_draws = (
    base_revenue * yield_multiplier * price_multiplier
    - farm.operating_costs * cost_multiplier
    - farm.fixed_costs
    + farm.insurance_income
    + farm.govt_payments
)
dscr_draws = (
    income_draws + farm.depreciation + farm.interest_expense - farm.owner_draws
) / farm.debt_obligations

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(income_draws, bins=40, color="#4C7A34", alpha=0.85, edgecolor="white")
axes[0].axvline(risk.p10_income, color="#A63A2B", linestyle="--", linewidth=2, label="P10 income")
axes[0].axvline(risk.p50_income, color="#1F4E79", linestyle="--", linewidth=2, label="P50 income")
axes[0].set_title("Simulated Net Farm Income")
axes[0].set_xlabel("Income ($)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].hist(dscr_draws, bins=40, color="#C69214", alpha=0.85, edgecolor="white")
axes[1].axvline(1.0, color="#A63A2B", linestyle="--", linewidth=2, label="DSCR = 1.0")
axes[1].axvline(risk.p10_dscr, color="#1F4E79", linestyle="--", linewidth=2, label="P10 DSCR")
axes[1].set_title("Simulated DSCR")
axes[1].set_xlabel("DSCR")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()


## 4. Takeaways

A lender or operator usually cares less about a single average result and more about downside resilience. The summary below turns the simulation output into a quick readout.


In [ ]:
downside_gap = risk.mean_income - risk.p10_income
coverage_label = "comfortable" if risk.p10_dscr >= 1.25 else "watchlist" if risk.p10_dscr >= 1.0 else "stressed"

takeaways = pd.DataFrame(
    [
        ["Downside income gap", f"${downside_gap:,.0f}", "Gap between average income and P10 income"],
        ["Default probability", f"{100 * risk.default_probability:.1f}%", "Probability that DSCR falls below 1.0"],
        ["Coverage posture", coverage_label, "Quick read on downside DSCR resilience"],
        ["Worst-case shortfall", f"${risk.worst_case_shortfall:,.0f}", "Largest simulated debt-service gap"],
    ],
    columns=["signal", "value", "why it matters"],
)

takeaways


## 5. Optional Connector Examples

The package also includes public-data connectors that can replace hard-coded assumptions with external yield, price, and weather inputs.

These examples are intentionally optional:

- the USDA NASS connector requires an API key
- the NASA POWER connector requires network access
- the main notebook flow above stays runnable without either one


In [ ]:
# USDA NASS requires an environment variable before use:
# export USDA_NASS_API_KEY="your-api-key"

from agfin.connectors import get_crop_price, get_crop_yield, get_weather_history

# Example only. Uncomment when you have API/network access.
# iowa_corn_yield = get_crop_yield("corn", "IA", 2022)
# iowa_corn_price = get_crop_price("corn", "IA", 2022)
# ames_weather = get_weather_history(lat=42.0308, lon=-93.6319, start_year=2020, end_year=2022)


## 6. Next Iteration Ideas

Natural ways to deepen this notebook from here:

- replace the hard-coded crop assumptions with live NASS lookups
- bring in NASA POWER weather history for scenario context
- compare a base case and a stressed leverage case side by side
- add a lender-oriented recommendation block based on DSCR thresholds
